# 🧠 SPoRC Dataset Preprocessing — Full Pipeline

**Steps Overview:**
1. Download SPoRC dataset  
2. Decompress `.jsonl.gz` files  
3. Preview dataset structure  
4. Convert speakerTurnData to CSV  
5. Check role distribution  
6. Filter host/guest roles  
7. Check empty / short utterances  
8. Remove empty or meaningless text entries

In [ ]:
# ==========================================================
# main.ipynb — SPoRC Dataset Preprocessing Full Pipeline
# ==========================================================
# Steps Overview:
#   1. Download SPoRC dataset
#   2. Decompress .jsonl.gz files
#   3. Preview dataset structure
#   4. Convert speakerTurnData to CSV
#   5. Check role distribution
#   6. Filter host/guest roles
#   7. Check empty / short utterances
#   8. Remove empty or meaningless text entries
# ==========================================================

import os
import pandas as pd
import numpy as np
import gzip
import json
from tqdm import tqdm
from datasets import load_dataset

# Define working directory
PROJECT_DIR = "/Users/allin1307/Desktop/semester 3/NLP/project"
os.makedirs(PROJECT_DIR, exist_ok=True)
print(f"Working directory: {PROJECT_DIR}")

# ==========================================================
# Step 1: Download dataset from Hugging Face
# ==========================================================

dataset = load_dataset("blitt/SPoRC")

episodes_path = os.path.join(PROJECT_DIR, "episodeLevelData.jsonl.gz")
turns_path = os.path.join(PROJECT_DIR, "speakerTurnData.jsonl.gz")

dataset["episodes"].to_json(episodes_path, orient="records", lines=True)
dataset["turns"].to_json(turns_path, orient="records", lines=True)

print(f"Downloaded to:\n{episodes_path}\n{turns_path}")

# ==========================================================
# Step 2: Decompress gzipped JSONL files into plain .jsonl
# ==========================================================

def decompress_gz(gz_path):
    jsonl_path = gz_path.replace(".gz", "")
    with gzip.open(gz_path, "rb") as f_in, open(jsonl_path, "wb") as f_out:
        f_out.write(f_in.read())
    print(f"Decompressed: {os.path.basename(jsonl_path)}")
    return jsonl_path

episodes_jsonl = decompress_gz(episodes_path)
turns_jsonl = decompress_gz(turns_path)

# ==========================================================
# Step 3: Preview structure of speaker-turn-level data
# ==========================================================
with open(turns_jsonl, "r") as f:
    first_lines = [json.loads(next(f)) for _ in range(5)]

print("Columns:", list(first_lines[0].keys()))
pd.DataFrame(first_lines)

# ==========================================================
# Step 4: Convert speakerTurnData.jsonl to CSV
# ==========================================================

csv_path = os.path.join(PROJECT_DIR, "sporc_turns_clean.csv")

with open(turns_jsonl, "r") as f_in, open(csv_path, "w", encoding="utf-8") as f_out:
    lines = [json.loads(line) for line in tqdm(f_in, desc="Converting")]
    df_turns = pd.DataFrame(lines)
    df_turns.to_csv(f_out, index=False)

print(f"Saved as CSV: {csv_path}")

# ==========================================================
# Step 5: Check and display role counts
# ==========================================================
assert os.path.exists(csv_path), "sporc_turns_clean.csv not found. Run Step 4 first."

# Load only the role column for speed/memory
df_roles = pd.read_csv(csv_path, usecols=["inferredSpeakerRole"])
role_counts = df_roles["inferredSpeakerRole"].value_counts(dropna=False)

print("Role distribution BEFORE filtering:\n", role_counts)
print("\nUnique role labels:", df_roles["inferredSpeakerRole"].unique())

# ==========================================================
# Step 6: Filter host/guest roles only
# ==========================================================
out_filtered = os.path.join(PROJECT_DIR, "sporc_turns_selected_clean.csv")

df_full = pd.read_csv(csv_path)

# Keep only host / guest
keep_labels = {"host", "guest"}
filtered = df_full[df_full["inferredSpeakerRole"].isin(keep_labels)].copy()
removed = len(df_full) - len(filtered)

filtered.to_csv(out_filtered, index=False)
print(f"✅ Filtered roles: kept {len(filtered):,}, removed {removed:,}")
print(f"Saved → {out_filtered}")

# (Optional) check distribution AFTER filtering
print("\nRole distribution AFTER filtering:\n", filtered["inferredSpeakerRole"].value_counts())

# ==========================================================
# Step 7: Count empty and short (<10 chars) utterances
# ==========================================================
empty_count = filtered["turnText"].isna().sum()
short_count = (filtered["turnText"].fillna("").str.len() < 10).sum()
print(f"Total: {len(filtered):,}")
print(f"Empty utterances: {empty_count:,}")
print(f"Short utterances (<10 chars): {short_count:,}")

# ==========================================================
# Step 8: Remove empty and meaningless short texts
# ==========================================================
df = filtered.copy()

# Remove NaN / empty / whitespace-only
df = df.dropna(subset=["turnText"])
df["turnText"] = df["turnText"].astype(str).str.strip()
df = df[df["turnText"] != ""]
df = df[~df["turnText"].str.fullmatch(r"[\.,!\?\-–—\s]+")]

output_final = os.path.join(PROJECT_DIR, "sporc_final_clean_min.csv")
df.to_csv(output_final, index=False)
print(f"✅ Final cleaned dataset saved: {output_final}")
